In [1]:
# Feature Engineering
#
# ├── 1. Feature Engineering for Machine Learning
# ├── 2. Features (X) and Target Variable (y)
# ├── 3. Feature Selection
# ├── 4. Encoding Categorical Features
# │      ├── Ordinal Encoding
# │      ├── Label Encoding
# │      ├── One-Hot Encoding
# │      └── Choosing the Appropriate Encoding Method
# ├── 5. Feature Scaling
# │      ├── StandardScaler
# │      └── MinMaxScaler
# ├── 6. Train-Test Split
# ├── 7. Applying StandardScaler
# ├── 8. Applying MinMaxScaler
# ├── 9. StandardScaler vs MinMaxScaler
# ├── 10. Algorithms Sensitive to Feature Scaling
# ├── 11. Choosing the Prepared Feature Matrix
# ├── 12. Final Machine Learning-Ready Data
# ├── 13. Final Verification
# └── 14. Dataset Ready for Machine Learning

#### **1. Feature Engineering for Machine Learning**

Feature Engineering is the process of transforming and preparing raw or cleaned data into features that can be effectively used by Machine Learning models.

After data cleaning and Exploratory Data Analysis (EDA), the next step is to prepare the dataset for Machine Learning.

In this notebook, we will:

- Identify features and the target variable.
- Select appropriate input features.
- Encode categorical features.
- Scale numerical features when required.
- Split the dataset into training and testing sets.
- Apply preprocessing correctly to avoid data leakage.

After these steps, the data will be ready for Machine Learning model development.

> **Important:** Preprocessing parameters such as scaling statistics must be learned from the training data only. The test data must remain unseen during preprocessing and model training.

In [2]:
# Import Required Libraries

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

In [3]:
# Load the cleaned dataset
df = pd.read_csv("../datasets/cleaned/student_performance_cleaned.csv", parse_dates=["admission_date"])

> Display Dataset Information

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   student_id             145 non-null    int64         
 1   full_name              145 non-null    str           
 2   gender                 145 non-null    str           
 3   age                    145 non-null    float64       
 4   department             145 non-null    str           
 5   semester               145 non-null    int64         
 6   marks                  145 non-null    float64       
 7   attendance_percentage  145 non-null    float64       
 8   city                   145 non-null    str           
 9   admission_date         139 non-null    datetime64[us]
 10  email                  145 non-null    str           
 11  scholarship            145 non-null    str           
 12  tuition_fee            145 non-null    float64       
 13  admission_year  

#### **2. Features and Target Variable**

##### **2.1 Identify Feature Types**
Now students can clearly identify:
- `Numerical Features`
- `Categorical Features`

Different feature types require different preprocessing techniques.

- Some Machine Learning algorithms are sensitive to feature scale, so numerical features may require scaling.
- Categorical features usually require encoding.
- Identifier columns should generally not be treated as predictive features.

In [5]:
# Display Numerical Features

df.select_dtypes(include="number").columns

Index(['student_id', 'age', 'semester', 'marks', 'attendance_percentage',
       'tuition_fee', 'admission_year', 'admission_month', 'admission_day'],
      dtype='str')

In [6]:
# Display Categorical Features

df.select_dtypes(include="str").columns

Index(['full_name', 'gender', 'department', 'city', 'email', 'scholarship',
       'admission_day_name'],
      dtype='str')

##### **2.2 Target Variable (y)**

In this dataset, `marks` is the target variable.

Because Marks is a continuous numerical value, this notebook is preparing the data for a **regression problem**.

The Machine Learning model will learn to predict Marks from the selected input features.

In [7]:
# Select the Target Variable

y = df["marks"]

#### **3. Features Selection**

Feature Selection is the process of selecting the most relevant input features for training a Machine Learning model.

Using only useful features can:

- Improve model performance.
- Reduce training time.
- Reduce model complexity.
- Minimize overfitting.

Not every column in a dataset should be used for model training.

In [8]:
feature_columns = [
    "age",
    "gender",
    "semester",
    "attendance_percentage",
    "department",
    "tuition_fee",
    "scholarship"
]

X = df[feature_columns].copy()

##### **3.1 Feature Data Types**

In [9]:
# Display Feature Data Types

X.dtypes

age                      float64
gender                       str
semester                   int64
attendance_percentage    float64
department                   str
tuition_fee              float64
scholarship                  str
dtype: object

#### **4. Encoding Categorical Features**

Most Machine Learning algorithms cannot work directly with text (categorical) data.

Therefore, categorical features must be converted into numerical values before training a Machine Learning model.

This process is called **Encoding**.

In [10]:
# Display Categorical Features

X.select_dtypes(include="str").columns

Index(['gender', 'department', 'scholarship'], dtype='str')

In [11]:
# Display Numerical Features

X.select_dtypes(include="number").columns

Index(['age', 'semester', 'attendance_percentage', 'tuition_fee'], dtype='str')

##### **4.1 Types of Categorical Data**

Categorical features are generally divided into two types:

> ##### **Ordinal Data**

Categories have a natural order.

Examples:

- Small → Medium → Large
- Low → Medium → High
- Poor → Average → Excellent

Ordinal data is usually encoded using **Ordinal Encoding**.

> ##### **Nominal Data**

Categories have no natural order.

Examples:

- Male, Female
- Computer Science, Mathematics, Physics
- Red, Blue, Green

Nominal data is usually encoded using **One-Hot Encoding**.

##### **4.2 Encoding Categorical Features**

Machine Learning algorithms work with numerical data. Therefore, categorical features must be converted into numerical values before training a model.

This process is called **Encoding**.

The appropriate encoding technique depends on whether the categorical variable is an input feature (`X`) or a target variable (`y`), and whether the categories have a meaningful order.

Common techniques include:

- **Ordinal Encoding** for ordered input features.
- **One-Hot Encoding** for nominal input features.
- **Label Encoding** mainly for categorical target labels (`y`) in classification problems.

### ***Ordinal Encoding vs Label Encoding***

Ordinal Encoding and Label Encoding can both convert categories into numerical values, but they are generally used for different purposes.

| Aspect | Ordinal Encoding | Label Encoding |
|---|---|---|
| Typical purpose | Input features (`X`) | Target variable (`y`) |
| Meaning | Represents ordered categories | Represents class labels |
| Scikit-learn class | `OrdinalEncoder` | `LabelEncoder` |
| Main use | Ordinal categorical features | Classification target |
| Example | Low → Medium → High | No → Yes |

#### **Ordinal Encoding**

Ordinal Encoding is used when the categories have a **meaningful order**.

For example:

```text
Low → 0
Medium → 1
High → 2
```

##### **4.2.1 Check Whether Features Are Nominal or Ordinal**

In [12]:
# Display Unique Values of Gender

X["gender"].unique()

<StringArray>
['male', 'female']
Length: 2, dtype: str

In [13]:
# Display Unique Values of Department

X["department"].unique()

<StringArray>
['SE', 'DS', 'CS', 'AI', 'IT']
Length: 5, dtype: str

In [14]:
# Display Unique Values of scholarship

X["scholarship"].unique()

<StringArray>
['Yes', 'No']
Length: 2, dtype: str

##### **4.3 Label Encoding and Ordinal Encoding**

**Label Encoding** is commonly used to convert categorical **target labels (`y`)** into numerical class labels in classification problems.

For example:

- No → 0
- Yes → 1

**Ordinal Encoding** is used for **ordinal input features (`X`)**, where the categories have a meaningful order.

For example:
```text
- Low → 0
- Medium → 1
- High → 2
```
> **Important:** The categorical features in our dataset are nominal, so we will not apply Ordinal Encoding or Label Encoding to these input features. We will use One-Hot Encoding instead.

##### **4.4 One-Hot Encoding**

One-Hot Encoding converts each category into a separate binary (0 or 1) column.

It is generally used for **Nominal Data**, where there is no natural order between categories.

Examples:

Department

- CS
- AI
- DS
- IT
- SE

becomes

- department_CS
- department_AI
- department_DS
- department_IT
- department_SE

Each row contains **1** for the corresponding category and **0** for all other categories.

In [15]:
X = pd.get_dummies(
    X,
    columns=["gender", "department","scholarship"],
    dtype=int
)

X.head()

,age,semester,attendance_percentage,tuition_fee,gender_female,gender_male,department_AI,department_CS,department_DS,department_IT,department_SE,scholarship_No,scholarship_Yes
0,20.0,4,80.8,15000.0,0,1,0,0,0,0,1,0,1
1,23.0,6,86.6,30000.0,0,1,0,0,1,0,0,1,0
2,18.0,1,85.1,50000.0,0,1,0,1,0,0,0,1,0
3,19.0,2,85.2,30000.0,0,1,0,0,1,0,0,1,0
4,18.0,1,87.1,25000.0,0,1,1,0,0,0,0,0,1


In [16]:
# Display Feature Names

X.columns

Index(['age', 'semester', 'attendance_percentage', 'tuition_fee',
       'gender_female', 'gender_male', 'department_AI', 'department_CS',
       'department_DS', 'department_IT', 'department_SE', 'scholarship_No',
       'scholarship_Yes'],
      dtype='str')

##### **4.5 Observation**

The Nominal features have been converted into multiple binary columns.

- 1 → The student belongs to that feature column.
- 0 → The student does not belong to that feature column.

One-Hot Encoding removes the artificial order that Label Encoding would introduce for nominal data.

##### **4.6 Choosing the Appropriate Encoding Method**

| Situation | Recommended Method |
|---|---|
| Ordinal categories | **Ordinal Encoding** |
| Nominal categories (3 or more) | **One-Hot Encoding** |
| Binary nominal categories (2 values only) | **Binary mapping or One-Hot Encoding** |
| Classification target (`y`) | **Label Encoding** |

#### **5. Feature Scaling**

Feature Scaling is the process of bringing numerical features to a similar scale.

In many datasets, different features have different ranges.

For example:

| Feature | Values |
|----------|---------|
| Age | 18 – 25 |
| Attendance Percentage | 60 – 100 |
| Tuition Fee | 15000 – 50,000 |
| Semester | 1 - 8 |

Because these features have very different ranges, Some distance-based, gradient-based, and regularization-sensitive algorithms can be affected when features have very different numerical scales.

Feature Scaling helps ensure that each feature contributes fairly during model training.

##### **5.1 Common Feature Scaling Techniques**

The two most commonly used feature scaling techniques are:

- `StandardScaler`
- `MinMaxScaler`

##### **5.2 StandardScaler**

`StandardScaler` standardizes numerical features so that they are centered around a mean of **0** with a standard deviation of approximately **1**.

StandardScaler is commonly used for Machine Learning algorithms that are sensitive to the scale of numerical features.

> **Important:** The scaler must be fitted using the **training data only**. The testing data must remain unseen while the preprocessing parameters are learned.

##### **5.3 Select Numerical Features for Scaling**

We will scale only the numerical features that are appropriate for scaling.

In our dataset, these features are:

- `age`
- `semester`
- `attendance_percentage`
- `tuition_fee`

Categorical features that were converted into 0/1 indicator columns will not be scaled.

In [17]:
# Select numerical features for scaling

numerical_features = [
    "age",
    "semester",
    "attendance_percentage",
    "tuition_fee"
]

numerical_features

['age', 'semester', 'attendance_percentage', 'tuition_fee']

#### **6. Train-Test Split**


##### **6.1 Why Split Before Feature Scaling?**

Before applying feature scaling, we first divide the dataset into training and testing sets.

This order is important because the test data must remain unseen while the preprocessing parameters are learned.

If we scale the complete dataset before splitting, the scaler calculates its parameters using both training and testing observations.

This causes **data leakage**.

##### **Correct Workflow**

```text
Original Dataset
       ↓
Feature Selection & Encoding
       ↓
Train-Test Split
       ↓
 ┌───────────────┐
 ↓               ↓
X_train         X_test
 ↓               ↓
Fit Scaler      Transform
 ↓
Transform
 ↓
Scaled X_train
       +
Scaled X_test
       ↓
Machine Learning

```


##### **6.2 Performing the Split**

We will use an **80:20 split**:

- 80% of the observations will be used for training.
- 20% will be used for testing.

The split is performed on the **unscaled feature matrix `X`**.

Scaling will be performed after the split.

In [18]:
# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (116, 13)
X_test : (29, 13)
y_train: (116,)
y_test : (29,)


##### **6.3 Understanding the Split**

The `train_test_split()` function creates four datasets:

| Variable | Description |
|----------|-------------|
| `X_train` | Training features |
| `X_test` | Testing features |
| `y_train` | Training target values |
| `y_test` | Testing target values |

The training data is used to learn the Machine Learning model.

The testing data is kept separate and is used later to evaluate how well the trained model performs on unseen data.

In [19]:
# Display the first five rows of the training features
X_train.head()

,age,semester,attendance_percentage,tuition_fee,gender_female,gender_male,department_AI,department_CS,department_DS,department_IT,department_SE,scholarship_No,scholarship_Yes
9,19.0,1,87.1,50000.0,0,1,0,0,0,1,0,1,0
4,18.0,1,87.1,25000.0,0,1,1,0,0,0,0,0,1
26,25.0,7,83.3,30000.0,1,0,0,0,0,0,1,1,0
120,18.0,1,89.7,50000.0,1,0,0,1,0,0,0,1,0
125,18.0,1,89.6,50000.0,0,1,0,0,1,0,0,1,0


In [20]:
# Display the first five rows of the testing features
X_test.head()

,age,semester,attendance_percentage,tuition_fee,gender_female,gender_male,department_AI,department_CS,department_DS,department_IT,department_SE,scholarship_No,scholarship_Yes
69,20.0,2,83.3,30000.0,0,1,0,0,1,0,0,1,0
140,18.0,1,87.1,50000.0,1,0,0,0,0,0,1,1,0
27,20.0,3,88.5,15000.0,0,1,0,0,0,1,0,0,1
19,24.0,6,87.1,30000.0,1,0,0,1,0,0,0,1,0
42,25.0,7,87.1,15000.0,0,1,0,0,0,1,0,0,1


In [21]:
# Display the first five training target values
y_train.head()

9      79.0
4      82.0
26     82.0
120    84.0
125    85.0
Name: marks, dtype: float64

In [22]:
# Display the first five testing target values
y_test.head()

69     73.0
140    82.0
27     86.0
19     79.0
42     90.0
Name: marks, dtype: float64

#### **7. Applying StandardScaler Correctly**

Now that the data has been divided into training and testing sets, we can apply StandardScaler.

The most important rule is:

> **Fit the scaler only on the training data.**

In [23]:
# Create the StandardScaler object
standard_scaler = StandardScaler()

# Fit the scaler using training data only
# Then transform the training data
X_train_scaled = X_train.copy()

X_train_scaled[numerical_features] = standard_scaler.fit_transform(
    X_train[numerical_features]
)

# Transform the testing data using the same fitted scaler
X_test_scaled = X_test.copy()

X_test_scaled[numerical_features] = standard_scaler.transform(
    X_test[numerical_features]
)

# Display the results
X_train_scaled.head()

,age,semester,attendance_percentage,tuition_fee,gender_female,gender_male,department_AI,department_CS,department_DS,department_IT,department_SE,scholarship_No,scholarship_Yes
9,-0.885407,-1.305038,0.188832,2.060412,0,1,0,0,0,1,0,1,0
4,-1.326210,-1.305038,0.188832,-0.260052,0,1,1,0,0,0,0,0,1
26,1.759414,1.320125,-0.201403,0.204041,1,0,0,0,0,0,1,1,0
120,-1.326210,-1.305038,0.455835,2.060412,1,0,0,1,0,0,0,1,0
125,-1.326210,-1.305038,0.445566,2.060412,0,1,0,0,1,0,0,1,0


##### **7.1 `fit_transform()` Vs `transform()`**

This is one of the most important concepts in Machine Learning preprocessing.

##### **What does `fit()` do?**

`fit()` learns the preprocessing parameters from the data.

For `StandardScaler`, these parameters are:

- Mean
- Standard deviation

##### **What does `transform()` do?**

`transform()` applies the already learned parameters to the data.

##### **What does `fit_transform()` do?**

`fit_transform()` performs both operations:

```text
fit()
  ↓
Learn parameters
  ↓
transform()
  ↓
Apply parameters
```

##### **7.2 Verifying StandardScaler**

We can use `describe()` to inspect the scaled training features.

After StandardScaler:

- The **mean should be approximately 0**.
- The **standard deviation should be approximately 1**.

We round the displayed results to two decimal places so that very small floating-point values are easier to understand.

##### **Training Data Verification**

In [24]:
# X_train_scaled[numerical_features].describe()

# Round the displayed values to two decimal places
X_train_scaled[numerical_features].describe().round(2)

,age,semester,attendance_percentage,tuition_fee
count,116.00,116.00,116.00,116.00
mean,0.00,0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00
min,-1.33,-1.31,-6.91,-1.19
25%,-0.89,-0.87,-0.06,-1.19
50%,-0.00,0.01,0.19,0.20
75%,0.88,0.88,0.41,0.20
max,1.76,1.76,0.99,2.06


##### **7.3 Testing Data After Scaling**

The test data is also scaled, but its mean and standard deviation do **not** have to be 0 and 1.

This is because the test data is transformed using the mean and standard deviation learned from the training data.

```text
X_train → fit_transform() → Mean ≈ 0, Std ≈ 1

X_test  → transform()      → Mean/Std are not necessarily 0/1

```

##### **Test Data Verification**

In [25]:
X_test_scaled[numerical_features].describe().round(2)

,age,semester,attendance_percentage,tuition_fee
count,29.00,29.00,29.00,29.00
mean,0.25,0.22,0.17,-0.34
std,0.80,0.82,0.38,0.95
min,-1.33,-1.31,-0.54,-1.19
25%,-0.44,-0.43,0.12,-1.19
50%,0.44,0.45,0.19,0.20
75%,0.88,0.88,0.30,0.20
max,1.76,1.76,0.93,2.06


##### **7.4 How StandardScaler Works**

StandardScaler performs two steps for every numerical feature.

##### Step 1

Subtract the **Mean** from every value.

```
Difference = Value − Mean
```

##### Step 2

Divide the result by the **Standard Deviation**.

```
Scaled Value = (Value − Mean) / Standard Deviation
```

As a result:

- Values **below the Mean** become **negative**.
- Values **equal to the Mean** become approximately **0**.
- Values **above the Mean** become **positive**.

StandardScaler does **not** change the order of the data. It only changes the scale.

##### **Example**


| Student | Age |
| ------- | --: |
| A       |  18 |
| B       |  20 |
| C       |  22 |
| D       |  24 |
| E       |  26 |

##### **Calculate Means**

- (18 + 20 + 22 + 24 + 26) / 5 = 22.
- So difference from the mean

| Age |   Difference |
| --: | -----------: |
|  18 | 18 − 22 = -4 |
|  20 | 20 − 22 = -2 |
|  22 |  22 − 22 = 0 |
|  24 | 24 − 22 = +2 |
|  26 | 26 − 22 = +4 |


##### **Now Divide by Standard Deviation**

Suppose Standard Deviation = **2.83**

| Age | Difference | Scaled Value |
| --: | ---------: | -----------: |
|  18 |         -4 |        -1.41 |
|  20 |         -2 |        -0.71 |
|  22 |          0 |         0.00 |
|  24 |          2 |         0.71 |
|  26 |          4 |         1.41 |

These are example standardized values produced by applying the StandardScaler formula.

StandardScaler learns the Mean and Standard Deviation for each numerical feature independently.


##### **7.5 Should We Scale Every Column?**

No.

We generally scale **appropriate numerical features**.

Categorical features that have been converted into binary indicator columns through One-Hot Encoding are usually left as `0` and `1`.

In our dataset:

##### Numerical features to scale

- `age`
- `semester`
- `attendance_percentage`
- `tuition_fee`

##### Categorical indicator features

- `gender_*`
- `department_*`
- `scholarship_*`

These remain as 0/1 values.

#### **8. MinMaxScaler**

`MinMaxScaler` transforms numerical features to a specified range.

By default, the range is:

- Minimum of the training data → approximately 0
- Maximum of the training data → approximately 1
- Other training values are scaled proportionally

Unlike StandardScaler, MinMaxScaler keeps the **training data** within the selected range.

> **Important:** When the scaler is fitted on training data and then applied to unseen test data, some test values may fall slightly below 0 or above 1 if they are outside the training range.

It is useful when a bounded numerical range is desirable.

##### **8.1 Applying MinMaxScaler**

In [26]:
# Create the MinMaxScaler object
minmax_scaler = MinMaxScaler()

# Fit on training data only and transform training data
X_train_minmax = X_train.copy()

X_train_minmax[numerical_features] = minmax_scaler.fit_transform(
    X_train[numerical_features]
)

# Transform testing data using the same fitted scaler
X_test_minmax = X_test.copy()

X_test_minmax[numerical_features] = minmax_scaler.transform(
    X_test[numerical_features]
)

# Display the first five rows
X_train_minmax.head()

,age,semester,attendance_percentage,tuition_fee,gender_female,gender_male,department_AI,department_CS,department_DS,department_IT,department_SE,scholarship_No,scholarship_Yes
9,0.142857,0.000000,0.898570,1.000000,0,1,0,0,0,1,0,1,0
4,0.000000,0.000000,0.898570,0.285714,0,1,1,0,0,0,0,0,1
26,1.000000,0.857143,0.849155,0.428571,1,0,0,0,0,0,1,1,0
120,0.000000,0.000000,0.932380,1.000000,1,0,0,1,0,0,0,1,0
125,0.000000,0.000000,0.931079,1.000000,0,1,0,0,1,0,0,1,0


##### **8.2 Verify MinMaxScaler**

We can inspect the numerical features after MinMax scaling.

The training data will have values between approximately `0` and `1`.

The testing data is transformed using the minimum and maximum values learned from the **training data**.

Therefore, some test values can occasionally fall slightly below 0 or above 1 if they are outside the training range.

This is normal and does not indicate an error.

In [27]:
X_train_minmax[numerical_features].describe()

,age,semester,attendance_percentage,tuition_fee
count,116.000000,116.000000,116.000000,116.000000
mean,0.429803,0.426108,0.874658,0.365764
std,0.325490,0.327927,0.127178,0.309156
min,0.000000,0.000000,0.000000,0.000000
25%,0.142857,0.142857,0.867035,0.000000
50%,0.428571,0.428571,0.898570,0.428571
75%,0.714286,0.714286,0.927178,0.428571
max,1.000000,1.000000,1.000000,1.000000


In [28]:
X_test_minmax[numerical_features].describe()

,age,semester,attendance_percentage,tuition_fee
count,29.000000,29.000000,29.000000,29.000000
mean,0.512315,0.497537,0.896417,0.261084
std,0.260402,0.268949,0.047525,0.293439
min,0.000000,0.000000,0.806242,0.000000
25%,0.285714,0.285714,0.889467,0.000000
50%,0.571429,0.571429,0.898570,0.428571
75%,0.714286,0.714286,0.912874,0.428571
max,1.000000,1.000000,0.992198,1.000000


##### **9. StandardScaler vs MinMaxScaler**

| StandardScaler | MinMaxScaler |
|---|---|
| Centers values around a mean of 0 | Scales values to a specified range |
| Standard deviation becomes approximately 1 | Default range is 0 to 1 |
| Can produce negative values | Training values normally remain between 0 and 1 |
| Often useful for algorithms sensitive to feature scale | Useful when a bounded feature range is desirable |
| Can be affected by outliers | Can be strongly affected by extreme minimum/maximum values |

Neither scaler is universally better. The appropriate choice depends on the dataset and the Machine Learning algorithm.

#### **10. Which Machine Learning Algorithms Are Sensitive to Feature Scaling?**

Some Machine Learning algorithms are strongly affected by the scale of the input features, while others are relatively insensitive to it.

| Algorithm | Scaling Usually Recommended? |
|---|:---:|
| Linear Regression | Sometimes |
| Logistic Regression | Often |
| K-Nearest Neighbors (KNN) | Yes |
| Support Vector Machine (SVM) | Yes |
| K-Means Clustering | Yes |
| Principal Component Analysis (PCA) | Yes |
| Neural Networks | Usually |
| Decision Tree | No |
| Random Forest | No |
| XGBoost | Usually No |

> **Important:** "Usually recommended" does not mean that scaling is mathematically mandatory in every situation. The appropriate preprocessing depends on the algorithm, dataset, and implementation.

#### **11. Which Prepared Feature Matrix Should We Use?**

The appropriate feature matrix depends on the Machine Learning algorithm.

For algorithms that are sensitive to feature scale, we can use either:

- `X_train_scaled` and `X_test_scaled` when using StandardScaler.
- `X_train_minmax` and `X_test_minmax` when using MinMaxScaler.

For algorithms that do not require scaling, we can use:

- `X_train`
- `X_test`

The important point is that the training and testing data must always receive the **same preprocessing treatment**, with preprocessing parameters learned from the training data only.

#### **12. Final Machine Learning-Ready Data**

At this stage, we have prepared three versions of our feature data:

##### **12.1 Unscaled Data**
`X_train`
`X_test`

##### **12.2 Standardized Data**
`X_train_scaled`
`X_test_scaled`

##### **12.3 Min-Max Scaled Data**
`X_train_minmax`
`X_test_minmax`

#### **13. Final verification**

In [29]:
print("Unscaled:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nStandardScaler:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)

print("\nMinMaxScaler:")
print("X_train_minmax:", X_train_minmax.shape)
print("X_test_minmax :", X_test_minmax.shape)

print("\nTargets:")
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Unscaled:
X_train: (116, 13)
X_test : (29, 13)

StandardScaler:
X_train_scaled: (116, 13)
X_test_scaled : (29, 13)

MinMaxScaler:
X_train_minmax: (116, 13)
X_test_minmax : (29, 13)

Targets:
y_train: (116,)
y_test : (29,)


#### **14. Dataset Ready for Machine Learning**

The dataset has now completed the main Feature Engineering and data-preparation steps:

- Feature Selection
- Categorical Encoding
- Train-Test Split
- Feature Scaling when required

Most importantly, scaling was performed **after the train-test split**.

The scalers were fitted using the training data only and then used to transform both training and testing data.

Therefore, the test data remains unseen while preprocessing parameters are learned.

##### **14.1 Final Workflow**

```text
Cleaned Dataset
      ↓
Feature Selection
      ↓
Categorical Encoding
      ↓
Train-Test Split
      ↓
 ┌─────────────────────────┐
 ↓                         ↓
X_train                   X_test
 ↓                         ↓
Fit Scaler                Transform
 ↓                         ↓
Transform                  │
 ↓                         │
X_train_scaled             │
                           ↓
                    X_test_scaled
             OR
        X_train_minmax
        X_test_minmax
               ↓
      Machine Learning Model
               ↓
         Model Evaluation

```